# wrangle / household / costs

**Intent**
- Gives constants from the same classified rows, so they reconcile, and must:
  - Give `household_costs_normal_annual_dollars`, the normal annual cost.
  - Give `household_costs_essential_dollars`, the part that cannot be cut.
  - Give `household_costs_discretionary_pct`, the discretionary share.
  - Give `household_costs_irregular_{expected,p95}_dollars`, the expected and one-in-twenty irregular cost.
  - Give `household_costs_growth_{low,base,high}_pct`, cost growth to 2070 in percent per year.
  - Let spending be projected and a bad year provisioned for.

**Data**
- Personal data is git-ignored in `data/` and `project/`, with synthetic placeholders committed.
- `training` is seventy two months of synthetic spending by category and class, with one irregular expense a year.
- `training` also holds ten years of synthetic inflation and cost growth around 3.24 percent.
- Both are synthetic until the `bank` table and the real series are loaded.
- The `Bank` workbook on Drive holds transactions from 2021-05-01, each with a `Category` that the `Classifier` maps to a bucket.
- The `bank` plugin writes nothing to Postgres yet.

**Status**
- Growth is a placeholder, a lookup of three numbers.
- It carries no evaluation, so the golden replay is the gate.

**Future**
- Overall: the work still to do before the first release:
  - Promote the target from local to served, then add the Register stage, once the limits are fixed against real data.
  - Add a `bank` table to `wrangle`, or a loader for the year files, and `query_postgres` to `collect_run`.
  - Explore spend by category, and flag one-offs above a robust threshold of the category median.
  - Classify categories as essential, committed, discretionary or irregular, and check them against past behaviour.
  - Train a compound Poisson per irregular category, with known schedules kept apart.
  - Model growth as inflation plus a structural premium, both mean reverting, and simulate to 2070.
  - Take compound rates, with low, base and high as percentiles, conditioned on ageing, energy and insurance drivers.
  - Compare five year growth with the five year inflation mean, as the history is under five years.
  - Fix the limits against real data, then register and review before any release.
  - Add `household/income` for Age Pension, super and rent.
  - Serve the numbers to the planning sheet through `wrangle` and an Apps Script function, which is not yet built.

## Setup ([6.7.1](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#671-setup))

In [ ]:
from mlmodels import setup_workspace

ws = setup_workspace("wrangle/household/costs")


GOLDEN = "household_costs_golden"

## Collect ([6.7.2](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#672-collect))

In [ ]:
from mlmodels import collect_run

CATEGORIES = {
    "groceries": ("essential", 24000.0),
    "utilities": ("essential", 9600.0),
    "insurance": ("committed", 7200.0),
    "school": ("committed", 7200.0),
    "hospitality": ("discretionary", 12000.0),
    "subscriptions": ("discretionary", 2400.0),
    "holidays": ("discretionary", 9600.0),
    "lifestyle": ("discretionary", 48000.0),
}
LIFESTYLE_DEVIATIONS = [-300.0, 200.0, -100.0, 300.0, -200.0, 0.0, 100.0, -300.0, 200.0, -100.0, 300.0, -100.0]
LUMPS = [12000.0, 30000.0, 8000.0, 15000.0, 52000.0, 9000.0]
INFLATION_DEVIATIONS = [-1.2, 0.8, -0.4, 1.0, -0.6, 0.2, 0.6, -1.0, 0.4, 0.2]
GROWTH_DEVIATIONS = [0.5, -0.7, 0.3, 0.9, -0.4, -0.8, 0.6, 0.2, -0.3, -0.3]


def spending():
    rows = {"month": [], "category": [], "class": [], "amount_dollars": []}
    for month in range(72):
        for category, (kind, annual) in CATEGORIES.items():
            rows["month"].append(month)
            rows["category"].append(category)
            rows["class"].append(kind)
            rows["amount_dollars"].append(annual / 12.0 + (LIFESTYLE_DEVIATIONS[month % 12] if category == "lifestyle" else 0.0))
        if month % 12 == 5:
            rows["month"].append(month)
            rows["category"].append("irregular")
            rows["class"].append("irregular")
            rows["amount_dollars"].append(LUMPS[month // 12])
    return rows


collect_run(ws, 1000, label="Seventy two months of synthetic spending by category and class", name="household_costs_training", source="builder", build=spending)
collect_run(
    ws,
    1000,
    label="Ten years of synthetic inflation and cost growth",
    name="household_costs_growth_training",
    source="builder",
    build=lambda: {
        "year": list(range(2016, 2026)),
        "inflation_pct": [round(3.24 + deviation, 4) for deviation in INFLATION_DEVIATIONS],
        "cost_growth_pct": [round(4.5 + deviation, 4) for deviation in GROWTH_DEVIATIONS],
    },
)

## Prepare ([6.7.3](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#673-prepare))

In [ ]:
from mlmodels import prepare_split

prepare_split(ws, training=ws.run(1000, "household_costs_training"))
print("transactions", ws.training.height, "annual", ws.runs[(1000, "household_costs_growth_training")].frame.height)

## Train ([6.7.5](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#675-train))

In [ ]:
import numpy as np

growth = ws.runs[(1000, "household_costs_growth_training")].frame
month = ws.training["month"].to_numpy()
kind = ws.training["class"].to_numpy()
amount = ws.training["amount_dollars"].to_numpy()
latest = month >= month.max() - 11
regular = kind != "irregular"
NORMAL = round(float(amount[regular & latest].sum()), 0)
ESSENTIAL = round(float(amount[np.isin(kind, ["essential", "committed"]) & latest].sum()), 0)
DISCRETIONARY = round(float(amount[(kind == "discretionary") & latest].sum() / amount[regular & latest].sum() * 100.0), 2)
years = month // 12
lumps = np.array([amount[(kind == "irregular") & (years == year)].sum() for year in np.unique(years)])
INFLATION = round(float(growth["inflation_pct"].mean()), 2)
GROWTH = round(float(growth["cost_growth_pct"].mean()), 2)
ws.params = dict(
    zip(
        ws.outputs,
        [NORMAL, ESSENTIAL, DISCRETIONARY, round(float(lumps.mean()), 0), round(float(np.percentile(lumps, 95)), 0), INFLATION, GROWTH, round(2 * GROWTH - INFLATION, 2)],
        strict=True,
    )
)
totals = np.array([amount[month == m].sum() for m in np.unique(month)])
median = float(np.median(totals))
mad = float(np.median(np.abs(totals - median)))
flagged = np.flatnonzero(totals > median + 5.0 * mad).tolist()
print("flagged months", flagged, "constants", ws.params)
assert flagged == sorted(set(month[kind == "irregular"].tolist())), "the robust rule and the irregular class disagree"
assert list(ws.params.values()) == [120000.0, 48000.0, 60.0, 21000.0, 46500.0, 3.24, 4.5, 5.76], "the fit did not recover the placeholder"

## Define ([6.7.6](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#676-define))

In [ ]:
from mlmodels import define_model

define_model(ws, model=1000, flavour="static", source=ws.params, label="Latest year of spending, irregular lumps, inflation and cost growth")

## Package ([6.7.8](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#678-package))

In [ ]:
from mlmodels import package_model

golden = {ws.inputs[0]: [1.0, None], "expect": ["ok", "error"]}
package_model(ws, 1000, golden, target={"kind": "local"})